# Section 6: Synthesis

*Duration: 15 minutes*

---

The code is done. The results are in. This section is not about running more cells. It is about interpreting what we built, deciding when it is worth the complexity, and connecting the work back to the principle that governed the entire lab series: **escalation of effort must be justified by evidence**.

No new API calls. No new tools. This is a facilitated discussion structured around three questions that matter in real engagements.

## 6.1 Load the Results

The cumulative data from the full lab — passive RAG baseline and agent loop results — is loaded below for reference during the discussion.

In [ ]:
import json

# Load all result sets for the cumulative view
with open("../prebuilt/eval_results.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)
baseline_results = baseline_data["results"]

with open("../prebuilt/agent_loop_results.json", "r", encoding="utf-8") as f:
    agent_data = json.load(f)
agent_results = agent_data["results"]

passive_total = sum(1 for r in baseline_results if r["classification"] == "pass")
agent_total = sum(1 for r in agent_results if r.get("agent_classification") == "pass")

print(f"Results loaded.")
print(f"  Passive RAG : {passive_total}/10")
print(f"  Agent Loop  : {agent_total}/10")

## 6.2 Discussion Anchor 1: When Does an Agent Loop Help?

Not every RAG deployment needs an agent loop. Most do not. The agent loop adds value in specific situations:

- **Multi-step questions** that require combining information from different parts of the corpus
- **Table lookups** where the retriever returns surrounding text but misses the specific row
- **Out-of-scope detection** where the correct answer is "I don't know" and the passive pipeline cannot produce that response

Think about a real customer engagement. What types of questions in that domain would benefit from an agent loop? What types would not?

> **Facilitator note:** Guide the group toward specificity. "Complex questions" is not a useful answer. "Questions where the user asks about a specific row in a pricing table and the retriever returns the paragraph above the table instead" is a useful answer. The more specific the failure mode, the easier it is to decide whether an agent loop is justified.

## 6.3 Discussion Anchor 2: Justifying Complexity

An agent loop adds moving parts: tool definitions, a dispatch layer, iteration logic, trace recording. Each part is a surface for bugs, latency, and maintenance cost. A passive pipeline is simpler, faster, and easier to debug.

The question is not "is the agent loop better?" The question is: **does the improvement justify the added complexity for this specific use case?**

Consider:

- A customer-facing FAQ bot that answers 95% of questions correctly with passive RAG. Does it need an agent loop for the remaining 5%?
- An internal compliance tool where a wrong answer triggers a regulatory review. Does it need an agent loop even if passive RAG scores 90%?
- A research assistant where users expect the system to say "I don't know" rather than guess. Does it need the `no_answer` tool?

The answer depends on the cost of a wrong answer, not on the accuracy number.

> **Facilitator note:** Push back on the instinct that more capability is always better. The Escalation Lab's core principle applies here too: complexity must be justified by evidence. If passive RAG is good enough for the use case, adding an agent loop is over-engineering.

## 6.4 Discussion Anchor 3: Where Does the Agent Loop Sit on the Escalation Ladder?

The Escalation Lab established a progression:

```
Baseline → Chunking → RAG → Best-of-N → Fine-tuning
```

The agent loop sits *after* RAG and *before* fine-tuning on this ladder. It changes the control structure without changing the model or the data. This is significant because:

- It is **cheaper** than fine-tuning — no training data, no compute cost, no model management
- It is **more targeted** than fine-tuning — it addresses specific failure modes (bad retrieval, out-of-scope) rather than general model capability
- It is **reversible** — you can remove the agent loop and fall back to passive RAG without retraining anything

The escalation ladder with the agent loop:

```
Baseline → Chunking → RAG → Agent Loop → Fine-tuning → Custom Model
```

The agent loop fills the gap between "retrieval is not enough" and "we need to retrain the model." In many engagements, it is the last stop before fine-tuning. In some, it eliminates the need for fine-tuning entirely.

> **Facilitator note:** The escalation ladder is the framing that ties this lab back to the Escalation Lab. Participants should leave with a clear mental model of where each technique sits and what evidence triggers the escalation to the next level.

## 6.5 Cumulative Results

The table below shows the full journey: every evaluation question, scored at each stage of the pipeline.

In [ ]:
# Print the cumulative results table across the full lab
print("Cumulative Results Across the Full Lab")
print("=" * 75)
print(f"{'ID':<6} {'Category':<22} {'Passive RAG':<14} {'Agent Loop':<12} {'Change'}")
print("-" * 75)

for br in baseline_results:
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    
    passive_ok = br["classification"] == "pass"
    agent_ok = ar and ar.get("agent_classification") == "pass"
    
    passive_display = "pass" if passive_ok else "FAIL"
    agent_display = "pass" if agent_ok else "FAIL"
    
    if not passive_ok and agent_ok:
        change = "FIXED"
    elif passive_ok and not agent_ok:
        change = "REGRESSED"
    elif passive_ok and agent_ok:
        change = "\u2014"
    else:
        change = "\u2014"
    
    print(f"{br['id']:<6} {br.get('category', ''):<22} {passive_display:<14} {agent_display:<12} {change}")

print("-" * 75)
print(f"{'Total':<6} {'':<22} {passive_total}/10{'':<9} {agent_total}/10")

In [ ]:
# Summary statistics
recovered = sum(
    1 for br in baseline_results
    for ar in agent_results
    if br["id"] == ar["id"]
    and br["classification"] != "pass"
    and ar.get("agent_classification") == "pass"
)

regressed = sum(
    1 for br in baseline_results
    for ar in agent_results
    if br["id"] == ar["id"]
    and br["classification"] == "pass"
    and ar.get("agent_classification") != "pass"
)

print("Summary")
print("=" * 40)
print(f"  Passive RAG baseline : {passive_total}/10")
print(f"  Agent Loop           : {agent_total}/10")
print(f"  Questions recovered  : {recovered}")
print(f"  Questions regressed  : {regressed}")
print(f"  Net improvement      : +{recovered - regressed}")
print()
print("The model did not change.")
print("The corpus did not change.")
print("The control structure changed.")

## 6.6 Closing

This lab started with a specific observation: 2 of 10 evaluation questions failed after every improvement the Escalation Lab could apply. Those failures survived better chunking, retrieval-augmented generation, Best-of-N sampling, and LoRA fine-tuning.

We did not add the agent loop because it sounded impressive. We added it because the evaluation in Section 1 showed that the architecture was the bottleneck, and we could point to exactly which questions failed and why:

- Questions where the retriever returned irrelevant chunks and the pipeline had no mechanism to detect that before answering
- Questions where the correct response was "I don't know" and the pipeline had no mechanism to produce that response
- Questions that required combining facts from multiple retrievals and the pipeline had no mechanism to iterate

The agent loop added three capabilities: evaluate retrieval quality, rewrite queries, and decline to answer. The model stayed the same. The retriever stayed the same. The tools are simple Python functions with JSON descriptions. The complexity is minimal and targeted.

That is the escalation principle in practice: we measured the failure, identified the layer responsible, and applied the minimum intervention that addressed it. Nothing more.

> **FIELD TAKEAWAY**
>
> An agent loop is not a default architecture. It is an escalation. You reach for it when evaluation shows that passive RAG fails in ways that better data and better prompts cannot fix. The evidence justifies the complexity. The 2x2 evaluation matrix (answer correctness x reasoning correctness) tells you whether the agent loop is solving the problem reliably or getting lucky. If you cannot point to specific questions that fail and explain why the passive pipeline cannot fix them, you do not need an agent loop yet.

---

> *"This lab reflects how successful AI engagements actually unfold: you don't reach for agents because they're exciting. You reach for them because you've measured a failure that simpler architectures cannot fix. And when you do reach for them, you measure again to make sure the fix is real."*